# Cibuco_Boriken — BirdCLEF+ 2026 Inference
**Team:** Cibuco_Boriken | **Backbone:** EfficientNet-B0 | **CFAR T=0.3** | **234 species**

In [ ]:
# ── Cell 1: Install dependencies ──
!pip install -q librosa audioread
print('Dependencies installed ✅')

In [ ]:
# ── Cell 2: Clone repo ──
!git clone https://github.com/drosadocastro-bit/cibuco-boriken
import sys
sys.path.insert(0, '/kaggle/working/cibuco-boriken')
print('Repo cloned ✅')

In [ ]:
# ── Cell 3: Imports + config ──
import os
import numpy as np
import pandas as pd
import torch
import librosa
from pathlib import Path
from tqdm import tqdm

# Paths
DATA_DIR   = Path('/kaggle/input/birdclef-2026')
MODEL_PATH = Path('/kaggle/input/cibuco-boriken-model/efficientnet_b0_33k_best.pt')
OUTPUT_PATH = Path('/kaggle/working/submission.csv')

# Config
SAMPLE_RATE = 32000
WINDOW_SEC  = 5.0
HOP_SEC     = 5.0    # no overlap → 12 windows/file → fits 90min CPU ✅
N_MELS      = 128
TEMPERATURE = 0.3    # CFAR Goldilocks zone
BATCH_SIZE  = 64
DEVICE      = 'cuda' if torch.cuda.is_available() else 'cpu'

print(f'Device:      {DEVICE}')
print(f'Window:      {WINDOW_SEC}s (no overlap)')
print(f'Temperature: {TEMPERATURE}')
print('Config ready ✅')

In [ ]:
# ── Cell 4: Load 234 species from taxonomy.csv ──
taxonomy_df = pd.read_csv(DATA_DIR / 'taxonomy.csv')
SPECIES     = taxonomy_df['primary_label'].tolist()
NUM_SPECIES = len(SPECIES)
print(f'Species from taxonomy.csv: {NUM_SPECIES}')  # should be 234
print(f'First 5: {SPECIES[:5]}')
print(f'Last  5: {SPECIES[-5:]}')

In [ ]:
# ── Cell 5: Load EfficientNet-B0 model ──
import torchvision.models as tv_models
import torch.nn as nn

class BirdClassifier(nn.Module):
    def __init__(self, num_classes, temperature=0.3):
        super().__init__()
        self.temperature = temperature
        backbone = tv_models.efficientnet_b0(weights=None)
        in_features = backbone.classifier[1].in_features
        backbone.classifier = nn.Identity()
        self.backbone = backbone
        self.classifier = nn.Linear(in_features, num_classes)

    def forward(self, x):
        if x.shape[1] == 1:
            x = x.repeat(1, 3, 1, 1)  # 1-ch mel → 3-ch
        features = self.backbone(x)
        return self.classifier(features)

    def predict(self, x):
        with torch.no_grad():
            logits = self.forward(x)
            return torch.sigmoid(logits / self.temperature)

# Model was trained on 206 species — we map outputs to 234 taxonomy species
# Load train species list to build mapping
train_df      = pd.read_csv(DATA_DIR / 'train.csv')
TRAIN_SPECIES = sorted(train_df['primary_label'].unique().tolist())
NUM_TRAIN_SPECIES = len(TRAIN_SPECIES)
print(f'Train species (model output size): {NUM_TRAIN_SPECIES}')

model = BirdClassifier(num_classes=NUM_TRAIN_SPECIES, temperature=TEMPERATURE)
checkpoint = torch.load(MODEL_PATH, map_location=DEVICE)
if isinstance(checkpoint, dict) and 'model_state_dict' in checkpoint:
    model.load_state_dict(checkpoint['model_state_dict'])
elif isinstance(checkpoint, dict) and 'state_dict' in checkpoint:
    model.load_state_dict(checkpoint['state_dict'])
else:
    model.load_state_dict(checkpoint)

model = model.to(DEVICE)
model.eval()
print(f'Model loaded on {DEVICE} ✅')

In [ ]:
# ── Cell 6: Audio utilities ──
def audio_to_mel(audio, sr=SAMPLE_RATE, n_mels=N_MELS):
    mel = librosa.feature.melspectrogram(
        y=audio, sr=sr, n_mels=n_mels,
        n_fft=1024, hop_length=320, fmin=50, fmax=14000
    )
    mel_db = librosa.power_to_db(mel, ref=np.max)
    mel_db = (mel_db - mel_db.min()) / (mel_db.max() - mel_db.min() + 1e-8)
    return mel_db.astype(np.float32)

def load_audio_windows(filepath, window_sec=WINDOW_SEC, hop_sec=HOP_SEC, sr=SAMPLE_RATE):
    try:
        audio, _ = librosa.load(filepath, sr=sr, mono=True)
    except Exception as e:
        print(f'Error loading {filepath}: {e}')
        return [], []

    window_samples = int(window_sec * sr)
    hop_samples    = int(hop_sec * sr)
    windows, end_times = [], []

    start = 0
    while start + window_samples <= len(audio):
        chunk = audio[start:start + window_samples]
        windows.append(chunk)
        end_times.append(int((start + window_samples) / sr))  # end time in seconds
        start += hop_samples

    # Last partial window — pad if needed
    if start < len(audio):
        chunk = audio[start:]
        chunk = np.pad(chunk, (0, window_samples - len(chunk)))
        windows.append(chunk)
        end_times.append(int((start + window_samples) / sr))

    return windows, end_times

print('Audio utilities ready ✅')

In [ ]:
# ── Cell 7: CFAR adaptive thresholding ──
def cfar_threshold(all_probs, k=2.0, floor=0.05, ceiling=0.95):
    mu    = all_probs.mean(axis=0)
    sigma = all_probs.std(axis=0)
    thresholds = np.clip(mu + k * sigma, floor, ceiling)
    return thresholds

print('CFAR ready ✅')

In [ ]:
# ── Cell 8: Run inference on all test soundscapes ──
soundscape_dir   = DATA_DIR / 'test_soundscapes'
soundscape_files = sorted(soundscape_dir.glob('*.ogg'))
print(f'Test soundscapes: {len(soundscape_files)}')

# Build train_species → index map for output alignment
train_species_idx = {sp: i for i, sp in enumerate(TRAIN_SPECIES)}

all_rows = []

for sf in tqdm(soundscape_files, desc='Inference'):
    file_id = sf.stem  # e.g. BC2026_Test_0001_S05_20250227_010002

    windows, end_times = load_audio_windows(str(sf))
    if not windows:
        continue

    # Mel features
    mels   = np.stack([audio_to_mel(w) for w in windows])  # (N, n_mels, T)
    mels   = mels[:, np.newaxis, :, :]                      # (N, 1, n_mels, T)
    mels_t = torch.from_numpy(mels).to(DEVICE)

    # Batch inference → probs over TRAIN_SPECIES (206)
    probs_list = []
    for i in range(0, len(mels_t), BATCH_SIZE):
        batch = mels_t[i:i+BATCH_SIZE]
        probs_list.append(model.predict(batch).cpu().numpy())
    train_probs = np.vstack(probs_list)  # (N_windows, 206)

    # Build submission rows — map 206 train species → 234 taxonomy species
    for idx, (end_time) in enumerate(end_times):
        # Row ID: filename_endtime
        # e.g. BC2026_Test_0001_S05_20250227_010002_20
        row_id = f'{file_id}_{end_time}'
        row    = {'row_id': row_id}

        # Map train species probs to taxonomy species
        # Species in taxonomy but not in train → prob = 0.0
        for sp in SPECIES:
            if sp in train_species_idx:
                row[sp] = float(train_probs[idx, train_species_idx[sp]])
            else:
                row[sp] = 0.0

        all_rows.append(row)

print(f'Total rows generated: {len(all_rows)} ✅')

In [ ]:
# ── Cell 9: Generate submission.csv ──
submission_df = pd.DataFrame(all_rows)

# Align to sample_submission.csv column order
sample_sub   = pd.read_csv(DATA_DIR / 'sample_submission.csv')
expected_cols = sample_sub.columns.tolist()

# Add any missing columns as 0
for col in expected_cols:
    if col not in submission_df.columns:
        submission_df[col] = 0.0

submission_df = submission_df[expected_cols].fillna(0.0)
submission_df.to_csv(OUTPUT_PATH, index=False)

print(f'Submission saved ✅')
print(f'Shape: {submission_df.shape}')
print(submission_df.head(3))

In [ ]:
# ── Cell 10: Validate submission ──
sample_sub = pd.read_csv(DATA_DIR / 'sample_submission.csv')
our_sub    = pd.read_csv(OUTPUT_PATH)

print('=== Submission Validation ===')
print(f'Expected rows:  {len(sample_sub)}')
print(f'Our rows:       {len(our_sub)}')
print(f'Expected cols:  {len(sample_sub.columns)}')
print(f'Our cols:       {len(our_sub.columns)}')
print(f'Columns match:  {set(sample_sub.columns) == set(our_sub.columns)}')
print(f'No NaN:         {our_sub.isna().sum().sum() == 0}')
print()
if set(sample_sub.columns) == set(our_sub.columns):
    print('✅ SUBMISSION VALID — ready to submit!')
else:
    missing = set(sample_sub.columns) - set(our_sub.columns)
    print(f'⚠️  Missing columns: {missing}')